# Callbacks: stream answers while the experiment runs

A callback receives every answer **as soon as it is generated**. Partial results therefore
survive a crash or an interrupted run, long runs can be monitored, and answers can go
wherever you need them.

| Callback | Output |
| --- | --- |
| `PrintTableCallback` | a compact table in the console |
| `PrintCallback` | verbose console output, one block per answer |
| `CSVCallback(path)` | one CSV row per answer, header written once |
| `JSONLCallback(path)` | one JSON object per line, with the whole question item |
| your own subclass of `Callback` | anything, for example a database (`examples/custom_callback.py`) |

**Requirements.** The run in this notebook uses the demo configuration with a real local
Hugging Face model, like [Basics.ipynb](Basics.ipynb): install
`"rupsycho[huggingface,notebook] @ git+https://github.com/julianschelb/rupsycho.git"`; the
model (`Qwen/Qwen2.5-0.5B-Instruct`, about 1 GB) is downloaded on first use. The
cells that run it carry the `skip-execution` tag and have no stored output. For an offline
run with a scripted stand-in model see [quickstart.ipynb](quickstart.ipynb).

In [1]:
from pathlib import Path

import rupsycho as rup
from rupsycho.callbacks import CSVCallback, JSONLCallback, PrintTableCallback

CONFIG = Path("data/bfi_demo_config.json")
if not CONFIG.exists():  # the notebook was opened from the repository root
    CONFIG = Path("examples") / CONFIG

experiment = rup.experiment_from_file(CONFIG)
print(experiment.name, "-", experiment.list_models())

Big Five Inventory example - ['Qwen/Qwen2.5-0.5B-Instruct']


## Choose the outputs

Callbacks **append** to their file, so a second run into the same file would mix two runs.
The cell therefore removes old files first. `output/` is a scratch folder next to this
notebook; it is not part of the repository.

In [2]:
OUTPUT = CONFIG.parent.parent / "output"
OUTPUT.mkdir(exist_ok=True)
csv_path = OUTPUT / "callbacks_answers.csv"
jsonl_path = OUTPUT / "callbacks_answers.jsonl"
for path in (csv_path, jsonl_path):
    path.unlink(missing_ok=True)

callbacks = [PrintTableCallback(), CSVCallback(csv_path), JSONLCallback(jsonl_path)]

## Run with callbacks

The table is printed row by row while the model works. `run()` returns a `RunSummary`; with
`on_error="warn"` (the default) a failed call is logged and the run continues. Failed calls
reach the callbacks with `answer=None`.

**This cell downloads and runs the model.**

In [ ]:
summary = experiment.run(callbacks=callbacks, on_error="warn")
print(summary)

## What the files look like

The repository ships sample result files in the format the callbacks write (created offline
by `make_sample_outputs.py` with a scripted stand-in model, hence the model id `scripted`).
The CSV has the columns `experiment_name`, `instruction_item_id`, `instruction_item`,
`model_id`, `profile_id`, `random_seed`, `time` (seconds per call) and `answer`:

In [3]:
import json

import pandas as pd

SAMPLE = Path("data/output/bfi_demo_config_output.csv")
if not SAMPLE.exists():
    SAMPLE = Path("examples") / SAMPLE

sample = pd.read_csv(SAMPLE)
print(list(sample.columns))
sample[["instruction_item_id", "model_id", "profile_id", "random_seed", "answer"]].head(4)

['experiment_name', 'instruction_item_id', 'instruction_item', 'model_id', 'profile_id', 'random_seed', 'time', 'answer']


,instruction_item_id,model_id,profile_id,random_seed,answer
0,0,scripted,Optimistic Persona,7,"{answer: ""4. Agree a little""}"
1,0,scripted,Conservative Persona,7,3
2,1,scripted,Optimistic Persona,7,"I would say ""2. Disagree a little""."
3,1,scripted,Conservative Persona,7,"As an AI, I do not have personal opinions on t..."


The JSONL file has one object per answer; `instruction_item` carries the question with its
attributes, so a row is self-contained:

In [4]:
first_row = json.loads(SAMPLE.with_suffix(".jsonl").read_text(encoding="utf-8").splitlines()[0])
print(list(first_row))
print(first_row["instruction_item"])

['experiment_name', 'instruction_item_id', 'instruction_item', 'model_id', 'profile_id', 'random_seed', 'time', 'answer']
{'question': 'I see myself as someone who is talkative.', 'reversed': False, 'answer_options': None, 'attributes': {'dimension': '1'}}


## Write your own callback

Subclass `Callback` and implement `save_answer`. It receives the experiment, the index and
the object of the question, the model id, the persona id, the seed, the generation time in
seconds and the answer (`None` for a failed call). Exceptions inside a callback are
reported as warnings and never stop the experiment.

In [5]:
from collections import Counter

from rupsycho.callbacks import Callback


class TallyCallback(Callback):
    """Count the answered and the failed calls per persona."""

    def __init__(self):
        self.answered, self.failed = Counter(), Counter()

    def save_answer(
        self,
        experiment,
        instruction_item_id,
        instruction_item,
        model_id,
        profile_id,
        random_seed,
        time,
        answer,
    ):
        (self.failed if answer is None else self.answered)[profile_id] += 1


# Use it like the built-in callbacks: experiment.run(callbacks=[..., TallyCallback()]).
# Here we feed it two calls by hand, the way the run loop does, to see what it does:
tally = TallyCallback()
item = experiment.questionnaire.instruction_items[0]
tally.save_answer(experiment, 0, item, "model", "Optimistic Persona", "7", 0.1, "3")
tally.save_answer(experiment, 0, item, "model", "Optimistic Persona", "7", 0.1, None)
print("answered:", dict(tally.answered), "failed:", dict(tally.failed))

answered: {'Optimistic Persona': 1} failed: {'Optimistic Persona': 1}


`examples/custom_callback.py` goes further: a callback that stores every answer in a SQLite
database, with a runnable offline demo.